[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C20_Frontier_Architectures_Course/05_ssm_mamba/05_ssm_mamba.ipynb)

# 05 · SSM 与 Mamba（从零实现）

目标：离散化连续 SSM，用**递推**与**卷积**两种方式各算一遍并验证等价，实现**并行扫描**，演示 Mamba 的**选择性**。

路线：离散化 → 递推 → 卷积等价 → 并行扫描 → 选择性 Δ → ✏️ 练习 → 🧪 复杂度账胶囊。

## 1 · 离散化：连续 (a,b) → 离散 (ā,b̄)

对角 SSM（每通道标量 a<0）的 ZOH 离散化。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def discretize(a, b, dt):
    # 对角连续 SSM 的 ZOH 离散化；a<0 保证稳定
    abar = np.exp(dt * a)
    bbar = (abar - 1.0) / a * b      # = (exp(dt a)-1)/a * b
    return abar, bbar

N = 4
a = -np.array([0.5, 1.0, 2.0, 4.0])   # N 个衰减率
b = np.ones(N)
abar, bbar = discretize(a, b, dt=0.1)
print('ā =', np.round(abar, 4), '(都在(0,1)，稳定)')
print('b̄ =', np.round(bbar, 4))
assert np.all((abar > 0) & (abar < 1))

## 2 · 递推视角（推理：O(1) 状态）

每步只更新固定大小的状态 h，与序列长度无关。

In [ ]:
def ssm_recurrent(x, abar, bbar, C):
    # x:(T,) 标量输入; 状态 h:(N,); 输出 y:(T,)
    T = len(x); N = len(abar)
    h = np.zeros(N); y = np.zeros(T)
    for t in range(T):
        h = abar * h + bbar * x[t]   # 更新固定大小状态
        y[t] = C @ h
    return y

C = rng.standard_normal(N)
x = rng.standard_normal(20)
y_rec = ssm_recurrent(x, abar, bbar, C)
print('递推输出形状', y_rec.shape)
assert y_rec.shape == (20,)

## 3 · 卷积视角（训练：可并行）

卷积核 `K[k] = Σ_n C_n · ā_n^k · b̄_n`，则 `y = x * K`（因果）。

In [ ]:
def ssm_kernel(abar, bbar, C, L):
    ks = np.arange(L)
    powers = abar[None, :] ** ks[:, None]      # (L, N)
    return (powers * (C * bbar)[None, :]).sum(axis=1)   # (L,)

def causal_conv(x, K):
    T = len(x); y = np.zeros(T)
    for t in range(T):
        kk = min(t + 1, len(K))
        y[t] = np.dot(K[:kk], x[t::-1][:kk])
    return y

K = ssm_kernel(abar, bbar, C, L=len(x))
y_conv = causal_conv(x, K)
print('递推 vs 卷积 最大误差:', round(np.abs(y_rec - y_conv).max(), 12))
assert np.allclose(y_rec, y_conv), '两种视角必须数值一致'
print('✅ 递推与卷积等价 —— SSM 既能高效推理又能并行训练的根基')

## 4 · 并行扫描（Hillis–Steele）

一阶线性递推 `h_t=a_t h_{t-1}+b_t` 的段变换可结合，可在 O(log T) 深度并行。

In [ ]:
def seq_scan(a, b):
    h = np.zeros_like(b, dtype=float); prev = 0.0
    for t in range(len(a)):
        prev = a[t] * prev + b[t]; h[t] = prev
    return h

def parallel_scan(a, b):
    A = a.astype(float).copy(); B = b.astype(float).copy()
    T = len(a); d = 1
    while d < T:
        A2, B2 = A.copy(), B.copy()
        for t in range(T - 1, d - 1, -1):
            B2[t] = A[t] * B[t - d] + B[t]
            A2[t] = A[t] * A[t - d]
        A, B = A2, B2
        d *= 2
    return B

a_t = rng.uniform(0.1, 0.9, 32)
b_t = rng.standard_normal(32)
assert np.allclose(seq_scan(a_t, b_t), parallel_scan(a_t, b_t)), '并行扫描应与串行一致'
print('✅ 并行扫描正确（Mamba 时变递推靠它在 GPU 上并行）')

## 5 · 选择性：让 Δ 随输入变化

Mamba 的核心：Δ（进而 ā,b̄）依赖输入，于是模型能按内容决定“记多久/记什么”。

In [ ]:
def selective_step(x, a, b, C, dt_base=0.1):
    # Δ_t = softplus(dt_base + 0.5*x_t)，随输入变化（简化标量门）
    T = len(x); N = len(a); h = np.zeros(N); y = np.zeros(T)
    for t in range(T):
        dt = np.log1p(np.exp(dt_base + 0.5 * x[t]))    # softplus，恒正
        abar = np.exp(dt * a); bbar = (abar - 1) / a * b
        h = abar * h + bbar * x[t]
        y[t] = C @ h
    return y

spike = np.zeros(20); spike[5] = 5.0      # 一个强输入
y_sel = selective_step(spike, a, b, C)
print('选择性 SSM 对强输入的响应（应在 t=5 后显著）:', np.round(y_sel[4:9], 3))
assert abs(y_sel[5]) > abs(y_sel[1]) + 1e-9

> 强输入（t=5）让 Δ 变大、把信息显著写入状态——这就是“按内容选择记什么”的雏形。LTI SSM 做不到这种输入相关的响应。

---
## ✏️ 练习 1：实现 ZOH 离散化

实现 `my_discretize(a, b, dt)`，验证 a<0 时 ā∈(0,1)、dt→0 时 b̄→dt·b。

In [ ]:
def my_discretize(a, b, dt):
    # TODO: 返回 (abar, bbar)，对角 ZOH
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
aa = -np.array([1.0, 2.0]); bb = np.ones(2)
ab, bb2 = my_discretize(aa, bb, 0.5)
assert np.all((ab > 0) & (ab < 1)), 'a<0 时 ā∈(0,1)'
ab_s, bb_s = my_discretize(aa, bb, 1e-4)
assert np.allclose(bb_s, 1e-4 * bb, rtol=1e-2), 'dt→0 时 b̄≈dt·b'
print('✅ 练习 1 通过')

## ✏️ 练习 2：递推与卷积等价

实现 `my_kernel(abar, bbar, C, L)`，使 `causal_conv(x, K)` 与 `ssm_recurrent` 一致。

In [ ]:
def my_kernel(abar, bbar, C, L):
    # TODO: K[k] = sum_n C_n * abar_n^k * bbar_n
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
KK = my_kernel(abar, bbar, C, len(x))
assert np.allclose(causal_conv(x, KK), ssm_recurrent(x, abar, bbar, C)), '卷积应等价递推'
print('✅ 练习 2 通过')

## ✏️ 练习 3：实现并行扫描

实现 `my_parallel_scan(a, b)`，与 `seq_scan` 一致。

In [ ]:
def my_parallel_scan(a, b):
    # TODO: 段变换 (a,b) 复合: (aj,bj)∘(ai,bi)=(aj*ai, aj*bi+bj)，Hillis-Steele
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
at = rng.uniform(0.1, 0.9, 16); bt = rng.standard_normal(16)
assert np.allclose(my_parallel_scan(at, bt), seq_scan(at, bt)), '应与串行扫描一致'
print('✅ 练习 3 通过')

---
### 📖 参考答案

In [ ]:
# 练习 1
def my_discretize(a, b, dt):
    abar = np.exp(dt * a)
    return abar, (abar - 1.0) / a * b

# 练习 2
def my_kernel(abar, bbar, C, L):
    ks = np.arange(L)
    powers = abar[None, :] ** ks[:, None]
    return (powers * (C * bbar)[None, :]).sum(1)

# 练习 3
def my_parallel_scan(a, b):
    A = a.astype(float).copy(); B = b.astype(float).copy()
    T = len(a); d = 1
    while d < T:
        A2, B2 = A.copy(), B.copy()
        for t in range(T-1, d-1, -1):
            B2[t] = A[t]*B[t-d] + B[t]; A2[t] = A[t]*A[t-d]
        A, B = A2, B2; d *= 2
    return B

---
## 🧪 真实数据胶囊：注意力 vs SSM 的复杂度账

比较两者随序列长度的计算/显存增长。

In [ ]:
def attn_cost(T, d):   return T*T*d, T*d        # 计算, KV cache(状态)
def ssm_cost(T, d, N): return T*d*N, d*N               # 计算(线性), 状态(固定)

print(f"{'T':>7} | {'Attn FLOPs':>14} {'Attn KV':>10} | {'SSM FLOPs':>14} {'SSM 状态':>10}")
for T in [1024, 8192, 65536]:
    af, ak = attn_cost(T, 64)
    sf, sk = ssm_cost(T, 64, 16)
    print(f'{T:7d} | {af:14d} {ak:10d} | {sf:14d} {sk:10d}')
print('\nT 增大时：注意力 FLOPs 二次增长、KV 线性增长；SSM FLOPs 线性、状态恒定。')

**🧪 胶囊练习**：实现 `crossover_T(d, N)`——求注意力 FLOPs 超过 SSM 的序列长度阈值（T²d > T·d·N → T > N）。

In [ ]:
def crossover_T(d, N):
    # TODO: 返回使 attn FLOPs > ssm FLOPs 的最小 T
    raise NotImplementedError

In [ ]:
assert crossover_T(64, 16) == 17   # T>N=16 -> 17
print('✅ 胶囊练习通过：当 T 超过状态维 N 后，SSM 的线性优势开始显现')

In [ ]:
# 📖 胶囊参考答案
def crossover_T(d, N):
    return N + 1

---
### 小结
- SSM 用固定状态的线性递推建模序列：递推(推理 O(1))与卷积(训练并行)等价。
- HiPPO/S4 解决长程记忆；Mamba 用选择性(时变 B/C/Δ)逼近注意力，靠并行扫描并行训练。
- 注意力 vs SSM 是精确检索 vs 长序列效率的权衡，混合架构两者兼取。

🎉 C20 现代架构课完结。回到 [课程主页](../index.html)。